In [ ]:
import time
import requests
import pandas as pd
import geopandas as gpd
from google.transit import gtfs_realtime_pb2

def fetch_bus_data():
    response = requests.get('https://gtfs.sofiatraffic.bg/api/v1/vehicle-positions')
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)
    
    bus_data = []
    for entity in feed.entity:
        route_id = entity.vehicle.trip.route_id
        occ = entity.vehicle.occupancy_status

        bus_data.append({
            'vehicle_id': entity.vehicle.vehicle.id,
            'latitude': entity.vehicle.position.latitude,
            'longitude': entity.vehicle.position.longitude,
            'raw_occupancy': occ if occ is not None else 0
            })
            
    return pd.DataFrame(bus_data)

# Constraints
total_runs = 24
runs_per_file = 3       # 3 runs, each 10 mins = 30 minutes
interval_minutes = 10

current_period_data = []

for i in range(total_runs):
    print(f"Collecting data: Run number {i + 1} of {total_runs}...")
    
    df_current = fetch_bus_data()
    if not df_current.empty:
        # Tag the data with the current time before appending
        df_current['timestamp'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
        current_period_data.append(df_current)
        
    if (i + 1) % runs_per_file == 0:
        file_index = (i + 1) // runs_per_file
        
        if current_period_data:
            df_combined = pd.concat(current_period_data, ignore_index=True)
            gdf = gpd.GeoDataFrame(
                df_combined, 
                geometry=gpd.points_from_xy(df_combined.longitude, df_combined.latitude),
                crs="EPSG:4326"
            )
            
            time_str = pd.Timestamp.now().strftime('%Y%m%d_%H%M')
            filename = f"Output_big/bus_occupancy_remainder_{time_str}.geojson"
            gdf.to_file(filename, driver="GeoJSON")
            print(f"Saved {filename} with {len(gdf)} total points.")
            
        else:
            print(f"No data found")
        current_period_data = []
        
    if i < total_runs - 1:
        print(f"Waiting {interval_minutes} minutes\n")
        time.sleep(interval_minutes * 60)

print("All data is collected")